# 06 — Backpropagation and Automatic Differentiation (PyTorch)

Backpropagation is an efficient application of the chain rule. Automatic differentiation does **not** replace the mathematics; it records the computational graph and evaluates the required derivatives for us.

For a parameter $w$:

$$
\frac{\partial \mathcal L}{\partial w}
$$

answers a local sensitivity question:

> If I make an infinitesimal increase in this parameter while holding the current computation fixed, how will the loss change?

The optimizer then decides how to use that gradient.


In [1]:
from pathlib import Path
import urllib.request
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)
MNIST_PATH = DATA_DIR / 'mnist.npz'
MNIST_URL = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz'
if not MNIST_PATH.exists():
    print('Downloading official MNIST archive...')
    urllib.request.urlretrieve(MNIST_URL, MNIST_PATH)
with np.load(MNIST_PATH) as data:
    x_train_raw, y_train_raw = data['x_train'], data['y_train']
    x_test_raw, y_test_raw = data['x_test'], data['y_test']
print('raw train:', x_train_raw.shape, y_train_raw.shape)
print('raw test :', x_test_raw.shape, y_test_raw.shape)


def balanced_subset(x, y, per_class, seed=SEED):
    rng = np.random.default_rng(seed)
    ids = []
    for cls in range(10):
        candidates = np.flatnonzero(y == cls)
        ids.extend(rng.choice(candidates, size=per_class, replace=False))
    ids = np.array(ids)
    rng.shuffle(ids)
    return x[ids], y[ids]

x_train, y_train = balanced_subset(x_train_raw, y_train_raw, 500)
x_test, y_test = balanced_subset(x_test_raw, y_test_raw, 100)
X_train = x_train.reshape(len(x_train), -1).astype('float32') / 255.0
X_test = x_test.reshape(len(x_test), -1).astype('float32') / 255.0
print('teaching train:', X_train.shape, y_train.shape)
print('teaching test :', X_test.shape, y_test.shape)
print('pixel range   :', float(X_train.min()), 'to', float(X_train.max()))


raw train: (60000, 28, 28) (60000,)
raw test : (10000, 28, 28) (10000,)
teaching train: (5000, 784) (5000,)
teaching test : (1000, 784) (1000,)
pixel range   : 0.0 to 1.0


## Manual derivative on one scalar neuron before using autodiff

Consider a sigmoid neuron with binary cross-entropy:

$$
z=wx+b,\qquad a=\sigma(z),\qquad
\mathcal L=-[y\log a+(1-y)\log(1-a)].
$$

For this common pairing,

$$
rac{\partial \mathcal L}{\partial z}=a-y,
$$

so by the chain rule,

$$
rac{\partial \mathcal L}{\partial w}=(a-y)x,\qquad
rac{\partial \mathcal L}{\partial b}=a-y.
$$

The next cell computes those numbers manually and verifies the weight gradient by finite differences. This is the mathematics that the framework later automates at scale.


In [2]:
x_scalar = 0.6
w_scalar = 0.8
b_scalar = -0.1
y_scalar = 1.0

z_scalar = w_scalar * x_scalar + b_scalar
a_scalar = 1.0 / (1.0 + np.exp(-z_scalar))
loss_scalar = -np.log(a_scalar)

dL_dz = a_scalar - y_scalar
dL_dw = dL_dz * x_scalar
dL_db = dL_dz

# Finite-difference check: perturb w by a tiny epsilon and estimate slope.
eps = 1e-5
z_plus = (w_scalar + eps) * x_scalar + b_scalar
a_plus = 1.0 / (1.0 + np.exp(-z_plus))
loss_plus = -np.log(a_plus)
finite_difference = (loss_plus - loss_scalar) / eps

print('z:', z_scalar)
print('activation a:', a_scalar)
print('loss:', loss_scalar)
print('analytic dL/dw:', dL_dw)
print('finite-difference dL/dw:', finite_difference)
print('analytic dL/db:', dL_db)


z: 0.38
activation a: 0.5938731029341427
loss: 0.5210896138659373
analytic dL/dw: -0.24367613823951434
finite-difference dL/dw: -0.24367570410355197
analytic dL/db: -0.40612689706585725


## What the framework code does

1. Build the network.
2. Run a mini-batch forward pass.
3. Compute scalar loss.
4. Ask autodiff for gradients.
5. Inspect gradient tensor shapes and norms.
6. Perform exactly one optimizer step.
7. Verify that a concrete parameter value changed.


In [3]:
import torch
import torch.nn as nn

torch.manual_seed(SEED)
model = nn.Sequential(
    nn.Linear(784, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)

# One mini-batch is enough to inspect the complete gradient graph.
X = torch.tensor(X_train[:32], dtype=torch.float32)
y = torch.tensor(y_train[:32], dtype=torch.long)

# Forward pass builds the autograd graph because model parameters require gradients.
logits = model(X)
loss = nn.CrossEntropyLoss()(logits, y)

# Clear any stale accumulated gradients, then run reverse-mode autodiff.
model.zero_grad(set_to_none=True)
loss.backward()

print('loss:', loss.item())
for name, parameter in model.named_parameters():
    print(
        f'{name:20s} '
        f'shape={tuple(parameter.shape)} '
        f'grad_norm={parameter.grad.norm().item():.6f}'
    )


loss: 2.34450101852417
0.weight             shape=(64, 784) grad_norm=0.777401
0.bias               shape=(64,) grad_norm=0.075652
2.weight             shape=(10, 64) grad_norm=0.224508
2.bias               shape=(10,) grad_norm=0.193347


In [4]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
first_weight_before = model[0].weight[0,0].item()
optimizer.step()
first_weight_after = model[0].weight[0,0].item()
print('parameter before:', first_weight_before)
print('parameter after :', first_weight_after)
print('delta           :', first_weight_after-first_weight_before)


parameter before: 0.02730494923889637
parameter after : 0.02730494923889637
delta           : 0.0


## Crucial distinction

- **Backpropagation** computes gradients.
- **Optimization** changes parameters.
- A forward pass does neither.

Confusing these three stages is one of the most common conceptual gaps in ANN learning.
